# MATH GR5360 Final Project — Notebook 3
## Performance Metrics & Extended (T, τ) Analysis

**Columbia University — Mathematical Methods in Financial Price Analysis**

**Group 1** — Primary: **TY** (10-Year US Treasury Futures) &nbsp;&nbsp;•&nbsp;&nbsp; Secondary: **BTC** (Bitcoin)

---

### Purpose

This notebook computes the full set of assignment-required performance metrics for the Channel-WithDDControl strategy at its best-in-sample $(L,S)$ parameters, then runs an **extended walk-forward grid** varying IS window $T$ (years) and OOS window $\tau$ (quarters) to quantify parameter-stability decay.

Toggle `MARKET_SELECT` to switch between `'TY'` and `'BTC'`.

In [ ]:
# ============================================================================
# MARKET SELECTOR
# ============================================================================
MARKET_SELECT = 'TY'     # 'TY' (primary, 10-Year Treasury Futures) or 'BTC' (secondary)

In [ ]:
# ============================================================================
# MARKET DATABASE + AUTO-CONFIG
# ============================================================================
MARKET_DATABASE = {
    # Ticker: (Name, Exchange, PV, Slippage, TickValue, PV_Multiplier)
    'BO': ('Soybean Oil',      'CBOT-CME',       600,  39,     6,     1),
    'DX': ('Dollar Index',     'NYBOT-ICE',      1000, 16.5,   5,     1),
    'HG': ('Copper',           'COMEX-NYMEX-CME', 250, 59.25, 12.5,   1),
    'HO': ('Heating Oil',      'NYMEX-CME',      420,  70.2,   4.2, 100),
    'JO': ('Orange Juice',     'NYBOT-ICE',      150, 183,     7.5,   1),
    'JY': ('Japanese Yen',     'CME',            1250, 53,     6.25,100),
    'SY': ('Soybeans',         'CBOT-CME',        50,  35.5,  12.5,   1),
    'SB': ('Sugar #11',        'NYBOT-ICE',     1120,  56.76, 11.2,   1),
    'SF': ('Swiss Franc',      'CME',           1250,  25.5,  12.5, 100),
    'TU': ('2-Year Treasury',  'CBOT-CME',      2000,  18.625,15.625, 1),
    'TY': ('10-Year Treasury', 'CBOT-CME',      1000,  18.625,15.625, 1),
    'WC': ('Wheat',            'CBOT-CME',        50,  30.5,  12.5,   1),
    'SM': ('Soybean Meal',     'CBOT-CME',       100,  57,    10,     1),
    'CC': ('Cocoa',            'NYBOT-ICE',       10, 103,    10,     1),
    'BZ': ('Schatz',           'EUREX',         1000,  10.5,   5,     1),
    'CL': ('Crude Oil WTI',    'NYMEX-CME',     1000,  46,    10,     1),
    'GC': ('Gold 100oz',       'COMEX-NYMEX-CME',100,  65,    10,     1),
    'SV': ('Silver',           'COMEX-NYMEX-CME',5000,243,    25,  0.01),
    'BTC':('Bitcoin',          'CME / Spot',       5,  50,    25,     1),
}

# Fallback if the selector cell wasn't executed yet — robust to out-of-order runs.
if 'MARKET_SELECT' not in globals():
    MARKET_SELECT = 'TY'

assert MARKET_SELECT in MARKET_DATABASE, f"Unknown market: {MARKET_SELECT}"
TICKER = MARKET_SELECT
_info = MARKET_DATABASE[TICKER]
MARKET = {
    'ticker':     TICKER,
    'name':       _info[0],
    'exchange':   _info[1],
    'PV':         _info[2],
    'slpg':       _info[3],
    'tick_value': _info[4],
    'E0':         100000,
}

print(f"MARKET: {MARKET['ticker']} — {MARKET['name']} ({MARKET['exchange']})  |  "
      f"PV=${MARKET['PV']}, Slpg=${MARKET['slpg']}, E0=${MARKET['E0']:,}")

In [ ]:
# ============================================================================
# IMPORTS + COLUMBIA THEME
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
from numba import jit
from tqdm.notebook import tqdm
from typing import Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Columbia visual identity
COLUMBIA_BLUE   = '#B9D9EB'
COLUMBIA_CORE   = '#75AADB'
COLUMBIA_NAVY   = '#012169'
COLUMBIA_DARK   = '#1D4F91'
COLUMBIA_ACCENT = '#C4D8E2'
COLUMBIA_GREY   = '#4B4B4B'
COLUMBIA_WARM   = '#E08119'
COLUMBIA_RED    = '#8B0000'

COLUMBIA_CMAP = LinearSegmentedColormap.from_list(
    'columbia', ['#FFFFFF', COLUMBIA_BLUE, COLUMBIA_CORE, COLUMBIA_DARK, COLUMBIA_NAVY])
COLUMBIA_DIVERGING = LinearSegmentedColormap.from_list(
    'columbia_div', [COLUMBIA_RED, COLUMBIA_WARM, '#FFFFFF', COLUMBIA_CORE, COLUMBIA_NAVY])

mpl.rcParams.update({
    'figure.figsize':      (14, 6),
    'figure.facecolor':    'white',
    'axes.facecolor':      'white',
    'axes.edgecolor':      COLUMBIA_NAVY,
    'axes.labelcolor':     COLUMBIA_NAVY,
    'axes.titlecolor':     COLUMBIA_NAVY,
    'axes.titleweight':    'bold',
    'axes.titlesize':      13,
    'axes.grid':           True,
    'grid.color':          COLUMBIA_ACCENT,
    'grid.alpha':          0.55,
    'grid.linewidth':      0.7,
    'xtick.color':         COLUMBIA_NAVY,
    'ytick.color':         COLUMBIA_NAVY,
    'legend.frameon':      True,
    'legend.edgecolor':    COLUMBIA_NAVY,
    'lines.linewidth':     1.3,
    'font.size':           11,
    'font.family':         'DejaVu Sans',
})

print(f"Theme: Columbia Blue  |  Market: {MARKET['ticker']} ({MARKET['name']})")

---
## 1. Load Results & Strategy

In [ ]:
# ============================================================================
# WALK-FORWARD RESULTS HANDOFF
# ============================================================================
# Since we don't persist intermediate CSVs, simply inherit `wf_results` from
# Notebook 02 if it was run in the same Jupyter kernel.  Otherwise fall back to
# a sensible default set of (L, S) below.
try:
    wf_results     # type: ignore[name-defined]
    if isinstance(wf_results, pd.DataFrame) and len(wf_results):
        print(f"Using wf_results from Notebook 02 ({len(wf_results)} periods)")
    else:
        raise NameError
except NameError:
    print("No wf_results found in kernel — will use default (L, S) below.")
    wf_results = None

In [ ]:
# Strategy function (same as Notebook 02)
@jit(nopython=True, cache=True)
def channel_strategy(Open, High, Low, Close, L, S, slpg, PV, E0, barsBack):
    N = len(Close)
    E = np.zeros(N) + E0
    DD = np.zeros(N)
    trades = np.zeros(N)
    HH = np.zeros(N)
    LL = np.zeros(N)
    for k in range(barsBack, N):
        HH[k] = np.max(High[k-L:k])
        LL[k] = np.min(Low[k-L:k])
    position = 0
    benchmarkLong = 0.0
    benchmarkShort = 0.0
    Emax = E0
    for k in range(barsBack, N):
        traded = False
        delta = PV * (Close[k] - Close[k-1]) * position
        if position == 0:
            buy = High[k] >= HH[k]
            sell = Low[k] <= LL[k]
            if buy and sell:
                delta = -slpg + PV * (LL[k] - HH[k])
                trades[k] = 1
            else:
                if buy:
                    delta = -slpg/2 + PV * (Close[k] - HH[k])
                    position = 1
                    traded = True
                    benchmarkLong = High[k]
                    trades[k] = 0.5
                if sell:
                    delta = -slpg/2 - PV * (Close[k] - LL[k])
                    position = -1
                    traded = True
                    benchmarkShort = Low[k]
                    trades[k] = 0.5
        elif position == 1 and not traded:
            sellShort = Low[k] <= LL[k]
            sell = Low[k] <= (benchmarkLong * (1 - S))
            if sellShort and sell:
                delta = delta - slpg - 2 * PV * (Close[k] - LL[k])
                position = -1
                benchmarkShort = Low[k]
                trades[k] = 1
            else:
                if sell:
                    delta = delta - slpg/2 - PV * (Close[k] - (benchmarkLong * (1 - S)))
                    position = 0
                    trades[k] = 0.5
                if sellShort:
                    delta = delta - slpg - 2 * PV * (Close[k] - LL[k])
                    position = -1
                    benchmarkShort = Low[k]
                    trades[k] = 1
            benchmarkLong = max(High[k], benchmarkLong)
        elif position == -1 and not traded:
            buyLong = High[k] >= HH[k]
            buy = High[k] >= (benchmarkShort * (1 + S))
            if buyLong and buy:
                delta = delta - slpg + 2 * PV * (Close[k] - HH[k])
                position = 1
                benchmarkLong = High[k]
                trades[k] = 1
            else:
                if buy:
                    delta = delta - slpg/2 + PV * (Close[k] - (benchmarkShort * (1 + S)))
                    position = 0
                    trades[k] = 0.5
                if buyLong:
                    delta = delta - slpg + 2 * PV * (Close[k] - HH[k])
                    position = 1
                    benchmarkLong = High[k]
                    trades[k] = 1
            benchmarkShort = min(Low[k], benchmarkShort)
        E[k] = E[k-1] + delta
        Emax = max(Emax, E[k])
        DD[k] = E[k] - Emax
    return E, DD, trades

In [ ]:
# ============================================================================
# LOAD DATA  (same loader as notebooks 01 & 02)
# ============================================================================
DATA_FILE = f'../data/{TICKER}-5minHLV.csv'

def load_data(filepath):
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip().str.lower()
    if 'date' in df.columns and 'time' in df.columns:
        df['datetime'] = pd.to_datetime(
            df['date'].astype(str) + ' ' + df['time'].astype(str), format='mixed')
    else:
        df['datetime'] = pd.to_datetime(df.iloc[:, 0])
    df.set_index('datetime', inplace=True)
    df.sort_index(inplace=True)
    rename = {}
    for c in df.columns:
        cl = c.lower()
        if   'open'  in cl: rename[c] = 'Open'
        elif 'high'  in cl: rename[c] = 'High'
        elif 'low'   in cl: rename[c] = 'Low'
        elif 'close' in cl: rename[c] = 'Close'
    df.rename(columns=rename, inplace=True)
    return df[['Open', 'High', 'Low', 'Close']].astype(float)

try:
    df = load_data(DATA_FILE)
    print(f"✓ Loaded {len(df):,} bars from {DATA_FILE}")
    print(f"  Range: {df.index.min()} → {df.index.max()} "
          f"({(df.index.max()-df.index.min()).days/365.25:.1f} yrs)")
except FileNotFoundError:
    print(f"⚠️  {DATA_FILE} not found — using synthetic series for {TICKER}")
    seed_map = {k: i for i, k in enumerate(MARKET_DATABASE.keys(), start=1)}
    np.random.seed(42 + seed_map[TICKER])
    n = 250_000
    dates = pd.date_range('2010-01-01', periods=n, freq='5min')
    start_prices = {'BO': 30, 'DX': 90, 'HG': 3, 'HO': 2, 'JO': 100, 'JY': 0.009,
                    'SY': 900, 'SB': 15, 'SF': 1, 'TU': 105, 'TY': 115, 'WC': 500,
                    'SM': 300, 'CC': 2500, 'BZ': 110, 'CL': 60, 'GC': 1300, 'SV': 15,
                    'BTC': 20000}
    base_sigma = 0.0015 if TICKER == 'BTC' else 0.0003
    returns = np.random.randn(n) * base_sigma \
              + np.sin(np.linspace(0, 8*np.pi, n)) * (base_sigma / 3)
    close = start_prices.get(TICKER, 100) * np.exp(np.cumsum(returns))
    df = pd.DataFrame({
        'Open':  close * (1 + np.random.randn(n) * base_sigma/3),
        'High':  close * (1 + np.abs(np.random.randn(n) * base_sigma)),
        'Low':   close * (1 - np.abs(np.random.randn(n) * base_sigma)),
        'Close': close,
    }, index=dates)
    df['High'] = df[['Open','High','Close']].max(axis=1)
    df['Low']  = df[['Open','Low', 'Close']].min(axis=1)
    print(f"✓ Generated {len(df):,} synthetic bars")

---
## 2. Performance Metrics (as required by assignment)

In [ ]:
def infer_bars_per_year(index):
    """Estimate number of 5-min bars per trading year from the data's own time axis."""
    total_bars = len(index)
    total_days = max((index.max() - index.min()).days, 1)
    bars_per_day = total_bars / total_days
    # crypto trades 24/7 (365 days); futures ~252 trading days
    cal_days_per_year = 365 if bars_per_day > 150 else 252
    return int(bars_per_day * cal_days_per_year)


def calculate_metrics(equity, trades, E0=100000, bars_per_year=19656):
    """
    Compute the full set of assignment-required metrics on the equity curve.

    Metrics: total profit, return %, annualised return and volatility, Sharpe,
    max drawdown (absolute & percent), return-on-account, trade count, win rate,
    average win / loss, win/loss ratio, profit factor.
    """
    final  = equity[-1]
    profit = final - E0

    # per-bar returns (strictly positive equity only)
    eq_pos  = equity[equity > 0]
    rets    = np.diff(eq_pos) / eq_pos[:-1]

    avg_ret = np.mean(rets)
    std_ret = np.std(rets)
    ann_ret = avg_ret * bars_per_year
    ann_std = std_ret * np.sqrt(bars_per_year)
    sharpe  = ann_ret / ann_std if ann_std > 0 else 0.0

    # drawdown
    peak     = np.maximum.accumulate(equity)
    dd       = equity - peak
    max_dd   = dd.min()
    max_dd_p = max_dd / peak[np.argmin(dd)] if peak[np.argmin(dd)] > 0 else 0.0

    # trades — only use full round-turns (trades == 1) for clean PnL bucketing
    pnl       = np.diff(equity)
    trade_ix  = np.where(trades[1:] > 0)[0]
    trade_pnl = pnl[trade_ix] if len(trade_ix) else np.array([0.0])
    winners   = trade_pnl[trade_pnl > 0]
    losers    = trade_pnl[trade_pnl < 0]

    n_trades = int(np.sum(trades))  # fractional trades summed to round-turns
    win_rate = len(winners) / max(len(winners) + len(losers), 1)
    avg_win  = np.mean(winners) if len(winners) else 0.0
    avg_loss = np.mean(losers)  if len(losers)  else 0.0
    gross_p  = np.sum(winners)  if len(winners) else 0.0
    gross_l  = -np.sum(losers)  if len(losers)  else 1.0
    pf       = gross_p / max(gross_l, 1)

    return {
        'Total Profit':       profit,
        'Return %':          (final / E0 - 1) * 100,
        'Ann. Return':        ann_ret * 100,
        'Ann. Volatility':    ann_std * 100,
        'Sharpe Ratio':       sharpe,
        'Max Drawdown $':     max_dd,
        'Max Drawdown %':     max_dd_p * 100,
        'Return on Account':  profit / abs(max_dd) if max_dd != 0 else 0.0,
        'Total Trades':       n_trades,
        'Win Rate %':         win_rate * 100,
        'Avg Winner':         avg_win,
        'Avg Loser':          avg_loss,
        'Win/Loss Ratio':     abs(avg_win / avg_loss) if avg_loss != 0 else 0.0,
        'Profit Factor':      pf,
    }

In [ ]:
# ============================================================================
# PICK BEST (L, S) AND RUN FULL-SAMPLE BACKTEST
# ============================================================================
DEFAULT_PARAMS = {
    'TY':  (5000, 0.02),
    'BTC': (2000, 0.03),
}

if wf_results is not None and len(wf_results) > 0:
    # Use the most-frequent best (L, S) pair from Notebook 02 walk-forward
    best_L = int(wf_results['L'].mode().iloc[0])
    best_S = float(wf_results['S'].mode().iloc[0])
    print(f"Selected from walk-forward: L={best_L}, S={best_S}")
else:
    best_L, best_S = DEFAULT_PARAMS.get(TICKER, (5000, 0.02))
    print(f"Using default: L={best_L}, S={best_S}")

bars_per_year = infer_bars_per_year(df.index)
print(f"Inferred bars-per-year for {TICKER}: {bars_per_year:,}")

barsBack = max(best_L + 1, 100)
E, DD, trades = channel_strategy(
    df['Open'].values, df['High'].values, df['Low'].values, df['Close'].values,
    best_L, best_S, MARKET['slpg'], MARKET['PV'], MARKET['E0'], barsBack
)

metrics = calculate_metrics(E, trades, MARKET['E0'], bars_per_year=bars_per_year)

In [ ]:
# ============================================================================
# DISPLAY PERFORMANCE METRICS + EQUITY/DD PLOT
# ============================================================================
print("\n" + "=" * 66)
print(f"PERFORMANCE METRICS — {TICKER} ({MARKET['name']})  "
      f"|  L={best_L}, S={best_S}")
print("=" * 66)

def _row(label, value, fmt):
    print(f"  {label:<22} {value:>{fmt}}")

print("\nReturns")
_row("Total Profit",    f"${metrics['Total Profit']:>14,.2f}", "15")
_row("Total Return",    f"{metrics['Return %']:>14.2f}%",      "15")
_row("Ann. Return",     f"{metrics['Ann. Return']:>14.4f}%",   "15")
_row("Ann. Volatility", f"{metrics['Ann. Volatility']:>14.4f}%","15")

print("\nRisk-Adjusted")
_row("Sharpe Ratio",    f"{metrics['Sharpe Ratio']:>14.4f}",   "15")
_row("Return on Acct.", f"{metrics['Return on Account']:>14.4f}","15")

print("\nDrawdown")
_row("Max DD ($)",      f"${metrics['Max Drawdown $']:>14,.2f}","15")
_row("Max DD (%)",      f"{metrics['Max Drawdown %']:>14.2f}%", "15")

print("\nTrade Statistics")
_row("Total Trades",    f"{metrics['Total Trades']:>14d}",     "15")
_row("Win Rate",        f"{metrics['Win Rate %']:>14.2f}%",    "15")
_row("Avg Winner",      f"${metrics['Avg Winner']:>14,.2f}",   "15")
_row("Avg Loser",       f"${metrics['Avg Loser']:>14,.2f}",    "15")
_row("Win/Loss Ratio",  f"{metrics['Win/Loss Ratio']:>14.4f}", "15")
_row("Profit Factor",   f"{metrics['Profit Factor']:>14.4f}",  "15")
print("=" * 66)

# --- Equity / DD visualisation --------------------------------------------
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                         gridspec_kw={'height_ratios': [2.2, 1]})

axes[0].plot(df.index, E, color=COLUMBIA_NAVY, linewidth=0.9)
axes[0].fill_between(df.index, E, MARKET['E0'],
                     where=E >= MARKET['E0'], color=COLUMBIA_CORE, alpha=0.25, interpolate=True)
axes[0].fill_between(df.index, E, MARKET['E0'],
                     where=E <  MARKET['E0'], color=COLUMBIA_WARM, alpha=0.25, interpolate=True)
axes[0].axhline(MARKET['E0'], color=COLUMBIA_GREY, linestyle='--', linewidth=1)
axes[0].set_title(f"{TICKER} ({MARKET['name']}) — Full-Sample Equity "
                  f"(L={best_L}, S={best_S})")
axes[0].set_ylabel('Equity ($)')

axes[1].fill_between(df.index, DD, 0, color=COLUMBIA_RED, alpha=0.45)
axes[1].plot(df.index, DD, color=COLUMBIA_RED, linewidth=0.7)
axes[1].set_title('Drawdown')
axes[1].set_ylabel('Drawdown ($)')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

In [ ]:
# Keep metrics available as a DataFrame in the kernel (no CSV written to disk)
metrics_df = pd.DataFrame([metrics])
metrics_df.insert(0, 'Ticker', TICKER)
metrics_df.insert(1, 'L', best_L)
metrics_df.insert(2, 'S', best_S)
metrics_df

---
## 3. Extended Analysis: Varying T and τ

In [ ]:
def run_wf_single(df, L_vals, S_vals, T, tau, slpg, PV, E0):
    """Run walk-forward with specific T and τ."""
    total_bars = len(df)
    total_days = (df.index.max() - df.index.min()).days
    bars_per_day = total_bars / total_days if total_days > 0 else 78
    bars_per_year = int(bars_per_day * 252)
    bars_per_quarter = int(bars_per_year / 4)
    
    IS_bars = T * bars_per_year
    OOS_bars = tau * bars_per_quarter
    
    results = []
    idx = 0
    
    while idx + IS_bars + OOS_bars <= total_bars:
        # IS optimization (simplified)
        best_obj = -np.inf
        best_is_profit = 0
        best_L, best_S = L_vals[0], S_vals[0]
        
        for L in L_vals:
            for S in S_vals:
                barsBack = max(int(L) + 1, 100)
                if IS_bars < barsBack + 100: continue
                E, DD, tr = channel_strategy(
                    df['Open'].values[idx:idx+IS_bars],
                    df['High'].values[idx:idx+IS_bars],
                    df['Low'].values[idx:idx+IS_bars],
                    df['Close'].values[idx:idx+IS_bars],
                    int(L), float(S), slpg, PV, E0, barsBack
                )
                profit = E[-1] - E[barsBack]
                max_dd = DD.min()
                obj = profit / abs(max_dd) if max_dd != 0 else 0
                if obj > best_obj:
                    best_obj = obj
                    best_is_profit = profit
                    best_L, best_S = int(L), float(S)
        
        # OOS
        barsBack = max(best_L + 1, 100)
        oos_start = idx + IS_bars
        oos_end = oos_start + OOS_bars
        if oos_end - oos_start < barsBack + 100:
            idx += OOS_bars
            continue
        
        E, DD, tr = channel_strategy(
            df['Open'].values[oos_start:oos_end],
            df['High'].values[oos_start:oos_end],
            df['Low'].values[oos_start:oos_end],
            df['Close'].values[oos_start:oos_end],
            best_L, best_S, slpg, PV, E0, barsBack
        )
        oos_profit = E[-1] - E[barsBack]
        oos_dd = DD.min()
        oos_obj = oos_profit / abs(oos_dd) if oos_dd != 0 else 0
        
        results.append({'is_obj': best_obj, 'oos_obj': oos_obj, 'oos_profit': oos_profit})
        idx += OOS_bars
    
    if len(results) == 0:
        return {'T': T, 'tau': tau, 'error': True}
    
    avg_is = np.mean([r['is_obj'] for r in results])
    avg_oos = np.mean([r['oos_obj'] for r in results])
    total_oos = np.sum([r['oos_profit'] for r in results])
    decay = avg_oos / avg_is if avg_is > 0 else 0
    
    return {'T': T, 'tau': tau, 'n': len(results), 'avg_is': avg_is, 'avg_oos': avg_oos, 'decay': decay, 'total_oos': total_oos, 'error': False}

In [ ]:
# ============================================================================
# EXTENDED ANALYSIS — walk-forward at multiple (T, τ) combinations
# ============================================================================
# Coarse (L, S) grid keeps runtime tractable; feel free to expand.
if TICKER == 'BTC':
    L_vals = np.array([1000, 2000, 3000, 4000])
    S_vals = np.array([0.01, 0.02, 0.03, 0.05])
else:
    L_vals = np.array([2000, 4000, 6000, 8000])
    S_vals = np.array([0.01, 0.02, 0.03, 0.04])

T_values   = [1, 2, 3, 4, 5, 6]    # In-sample years
tau_values = [1, 2, 3, 4]          # Out-of-sample quarters

print(f"Extended sweep — {TICKER}")
print(f"  L grid: {L_vals.tolist()}  |  S grid: {[round(s,3) for s in S_vals]}")
print(f"  T ∈ {T_values}  |  τ ∈ {tau_values}")
print(f"  Total combinations: {len(T_values)*len(tau_values)}\n")

ext_results = []
for T in tqdm(T_values, desc="T"):
    for tau in tau_values:
        res = run_wf_single(df, L_vals, S_vals, T, tau,
                            MARKET['slpg'], MARKET['PV'], MARKET['E0'])
        ext_results.append(res)

ext_df = pd.DataFrame(ext_results)
valid  = ext_df[~ext_df['error']].copy()
print(f"\nCompleted {len(valid)} / {len(ext_df)} valid (T, τ) combinations")

In [ ]:
# Display extended analysis
if len(valid) > 0:
    print("\n" + "=" * 66)
    print(f"EXTENDED ANALYSIS: T (IS years) vs τ (OOS quarters) — {TICKER}")
    print("=" * 66)
    print(valid[['T', 'tau', 'n', 'avg_is', 'avg_oos', 'decay', 'total_oos']]
          .to_string(index=False, float_format=lambda v: f"{v:>10.4f}"))
    best = valid.loc[valid['decay'].idxmax()]
    print(f"\nBest decay:  T={int(best['T'])} yr,  τ={int(best['tau'])} Q  "
          f"→ decay={best['decay']:.1%},  total OOS=${best['total_oos']:,.0f}")
else:
    print("No valid (T, τ) combinations — data span is too short for the chosen T/τ.")

In [ ]:
# ============================================================================
# VISUALIZE DECAY SURFACE (Columbia theme)
# ============================================================================
if len(valid) > 0:
    pivot_decay     = valid.pivot(index='T', columns='tau', values='decay')
    pivot_total_oos = valid.pivot(index='T', columns='tau', values='total_oos')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    # --- Decay heatmap ------------------------------------------------
    ax1 = axes[0]
    im1 = ax1.imshow(pivot_decay.values, cmap=COLUMBIA_CMAP,
                     aspect='auto', vmin=0, vmax=1)
    ax1.set_xticks(range(len(pivot_decay.columns)))
    ax1.set_xticklabels([f"{t}Q" for t in pivot_decay.columns])
    ax1.set_yticks(range(len(pivot_decay.index)))
    ax1.set_yticklabels([f"{t}Y" for t in pivot_decay.index])
    ax1.set_xlabel('τ (OOS quarters)')
    ax1.set_ylabel('T (IS years)')
    ax1.set_title(f'{TICKER}: OOS/IS Decay')
    plt.colorbar(im1, ax=ax1, label='decay = avg OOS obj / avg IS obj')
    for i in range(len(pivot_decay.index)):
        for j in range(len(pivot_decay.columns)):
            val = pivot_decay.iloc[i, j]
            if np.isfinite(val):
                txt_color = 'white' if val < 0.4 else COLUMBIA_NAVY
                ax1.text(j, i, f"{val:.2f}", ha='center', va='center',
                         fontsize=11, color=txt_color, fontweight='bold')

    # --- Total OOS profit heatmap ------------------------------------
    ax2 = axes[1]
    vmax = max(abs(pivot_total_oos.values.min()), abs(pivot_total_oos.values.max()))
    im2 = ax2.imshow(pivot_total_oos.values / 1000, cmap=COLUMBIA_DIVERGING,
                     aspect='auto', vmin=-vmax/1000, vmax=vmax/1000)
    ax2.set_xticks(range(len(pivot_total_oos.columns)))
    ax2.set_xticklabels([f"{t}Q" for t in pivot_total_oos.columns])
    ax2.set_yticks(range(len(pivot_total_oos.index)))
    ax2.set_yticklabels([f"{t}Y" for t in pivot_total_oos.index])
    ax2.set_xlabel('τ (OOS quarters)')
    ax2.set_ylabel('T (IS years)')
    ax2.set_title(f'{TICKER}: Total OOS Profit ($K)')
    plt.colorbar(im2, ax=ax2, label='Total OOS Profit ($K)')
    for i in range(len(pivot_total_oos.index)):
        for j in range(len(pivot_total_oos.columns)):
            val = pivot_total_oos.iloc[i, j] / 1000
            if np.isfinite(val):
                ax2.text(j, i, f"{val:+.0f}", ha='center', va='center',
                         fontsize=10, color=COLUMBIA_NAVY, fontweight='bold')

    plt.suptitle(f"{TICKER} ({MARKET['name']}) — Extended (T, τ) Analysis",
                 fontsize=14, color=COLUMBIA_NAVY, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Keep extended results as a DataFrame in the kernel (no CSV written)
if len(valid) > 0:
    valid  # display inline


---
## 4. Final Summary

In [ ]:
print("\n" + "=" * 72)
print(f"FINAL SUMMARY — {TICKER} ({MARKET['name']})")
print("=" * 72)

print(f"  Strategy:        Channel WithDDControl")
print(f"  Best (L, S):     L={best_L}, S={best_S}")
print(f"  Total Profit:    ${metrics['Total Profit']:,.2f}")
print(f"  Sharpe Ratio:    {metrics['Sharpe Ratio']:.4f}")
print(f"  Max Drawdown:    ${abs(metrics['Max Drawdown $']):,.2f}  "
      f"({metrics['Max Drawdown %']:.2f}%)")
print(f"  Win Rate:        {metrics['Win Rate %']:.1f}%")
print(f"  Profit Factor:   {metrics['Profit Factor']:.2f}")
print(f"  Return on Acct:  {metrics['Return on Account']:.4f}")

if len(valid) > 0:
    best = valid.loc[valid['decay'].idxmax()]
    print()
    print(f"  Recommended T:   {int(best['T'])} years")
    print(f"  Recommended τ:   {int(best['tau'])} quarter(s)")
    print(f"  Expected Decay:  {best['decay']:.1%}")

print("=" * 72)

---
## Next Steps

1. **Rerun for Secondary Market** — flip `MARKET_SELECT = 'BTC'` at the top and re-execute all three notebooks to obtain the parallel analysis on Bitcoin.
2. **C Implementation** — port `channel_strategy` for presentation-grade runtime on the full 951×96 grid.
3. **Presentation** — assemble a 30–40 minute deck with Columbia branding.

**End of Notebook 03**